# Deploy `proc_run_bronze_scd1` with `spark.sql()`

This notebook creates the procedure `${catalog}.utils.proc_run_bronze_scd1`.

## What it does
- calls `proc_prepare_target`
- reloads metadata with `proc_load_config`
- decides merge strategy (`PK` or `HASH`)
- builds and executes a generic SCD1 merge

## Expected utility functions
- `fn_normalize_identifier`
- `fn_has_pk`
- `fn_merge_strategy`
- `fn_contract_columns`


In [0]:
env = "dev"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS platform_{env}.utils")
print(f"Schema ensured: platform_{env}.utils")


In [0]:
spark.sql(f'''
CREATE OR REPLACE PROCEDURE platform_{env}.utils.proc_run_bronze_scd1
(
  IN p_config_name STRING,
  IN p_layer       STRING,
  IN p_system     STRING,
  IN p_full        STRING DEFAULT 'false'
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
  -- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
  -- Declare now variable with current timestamp
  DECLARE now TIMESTAMP DEFAULT current_timestamp();

  -- Declare variable for fully qualified target table name
  DECLARE target_table_fqn STRING DEFAULT '';

  -- Declare variable for columns string, default to '*'
  DECLARE columns STRING DEFAULT '*';
  -- Declare variable for query string
  DECLARE base_query STRING DEFAULT '';
  DECLARE hash_query STRING DEFAULT '';
  DECLARE query STRING DEFAULT '';
  -- Declare variable for where clause
  DECLARE where STRING DEFAULT '';
  DECLARE watermark STRING DEFAULT '';
  DECLARE state_type STRING DEFAULT '';
  DECLARE last_value STRING DEFAULT '';

  -- Declare variable for DDL schema statement
  DECLARE ddl_schema STRING DEFAULT '';
  -- Declare variable for DDL table statement
  DECLARE ddl_table STRING DEFAULT '';
  -- Declare variable for DML table statement
  DECLARE dml_table STRING DEFAULT '';
  DECLARE dml_watermark STRING DEFAULT '';

  -- Declare variable for DDL temp view
  DECLARE ddl_temp_view STRING DEFAULT '';

  DECLARE dml_max BIGINT DEFAULT 0;
  DECLARE dml_count BIGINT DEFAULT 0;
  DECLARE dml_range BIGINT DEFAULT 0;
  DECLARE step BIGINT DEFAULT 0;
  DECLARE dml_count_query STRING DEFAULT '';

  -- 1) Prepare target structure
--   CALL platform_{env}.utils.proc_prepare_target(p_config_name, p_layer, p_system);

  -- 2) Reload config into cfg
  CALL platform_{env}.utils.proc_load_config(p_config_name, p_layer, p_system);

  -- DROP TABLE IF EXISTS platform_{env}.control.tmp_result;
  CREATE OR REPLACE TABLE platform_{env}.control.tmp_result (
    config_id STRING,
    columns STRING,
    query STRING, 
    ddl_table STRING, 
    dml_table STRING, 
    dml_watermark STRING,
    dml_count STRING
  ) USING DELTA;

  -- Iterate over each row in configs
  iterConf: FOR row AS (SELECT * FROM cfg) DO

    -- Check if source_format is 'FEDERATION'
    CASE WHEN upper(row.cfg_source.source_format) = 'FEDERATION' THEN

      -- If target_schema_definition is not null, build columns string
      IF row.cfg_target.target_schema_definition IS NOT NULL THEN
        -- Set columns using target_schema_definition_sql
        SET columns = row.cfg_target.target_schema_definition_sql;

        SET base_query = concat(
          'SELECT ', 
          columns, 
          ' FROM ', 
          row.cfg_source.source_catalog, '.', 
          row.cfg_source.source_schema, '.', 
          row.cfg_source.source_object
        );

      ELSEIF row.cfg_source.source_query IS NOT NULL THEN
        SET base_query = row.cfg_source.source_query;
        -- Usado temporariamente para corrigir o campo "source_query"
        SET base_query = regexp_replace(base_query, '(?i)FROM public.', concat('FROM ', row.cfg_source.source_catalog, '.public.'));
        SET base_query = replace(base_query, ';', '');

      ELSE
        -- Build SELECT query string
        SET base_query = concat(
          'SELECT * FROM ', 
          row.cfg_source.source_catalog, '.', 
          row.cfg_source.source_schema, '.', 
          row.cfg_source.source_object
        );

      END IF;


      -- If load_mode is WATERMARK, build where clause
      IF upper(row.cfg_source.load_mode) = 'WATERMARK' THEN

        IF row.cfg_state.last_value IS NOT NULL AND lower(p_full) <> 'true' THEN
          SET last_value = row.cfg_state.last_value;
        ELSEIF row.cfg_source.watermark_initial_value IS NOT NULL THEN
          SET last_value = row.cfg_source.watermark_initial_value;
        END IF;

        IF row.cfg_state.state_type IS NOT NULL THEN
          SET state_type = row.cfg_state.state_type;
        ELSE
          SET state_type = "WATERMARK";
        END IF;

        IF state_type = 'WATERMARK' THEN
         
          IF row.cfg_source.lookback_days IS NOT NULL AND row.cfg_source.lookback_days > 0 THEN
            SET watermark = '
              BETWEEN 
              TO_TIMESTAMP("' || last_value || '")
              AND "' || now  ||'" - INTERVAL ' || row.cfg_source.lookback_days || ' DAYS
            ';
            SET last_value = now;
            
          END IF;

        -- ELSEIF state_type = 'ID' THEN
        --     SET watermark = concat(
        --       ' > ',
        --       last_value 
        --     );

        --     SET last_value = (SELECT MAX(CAMPO_X)....);

        END IF;

        SET where = concat(
          ' WHERE ', 
          row.cfg_source.watermark_column, 
          watermark
        );

        SET dml_watermark = concat(
            'MERGE INTO platform_{env}.control.cfg_ingestion_state AS T ',
            'USING ( ',
              'SELECT  ',
                '"',row.cfg_config.config_id, '" AS config_id, ',
                '"',state_type,'" AS state_type, ',
                'TO_TIMESTAMP("', now, '") - INTERVAL ', row.cfg_source.lookback_days, ' DAYS AS last_value, ',
                '"', now, '" AS last_run_ts, ',
                '"SUCCESS" AS status,  ',
                'NULL AS error_message ',
            ') AS S ON ( ',
              'T.config_id = S.config_id ',
            ') ',
            'WHEN MATCHED THEN UPDATE SET ',
              'T.last_value = S.last_value, ',
              'T.last_run_ts = S.last_run_ts ',
            'WHEN NOT MATCHED THEN INSERT (config_id, state_type, last_value, last_run_ts, status, error_message) VALUES ( ',
              'S.config_id, ',
              'S.state_type, ',
              'S.last_value, ',
              'S.last_run_ts, ',
              'S.status, ',
              'S.error_message ',
            ') '
        );

      END IF;

      -- Append where clause to query
      SET base_query = concat(base_query, where);

      -- Build statement to add _ingestion_bronze and _hashkey columns
      SET hash_query = concat(
        'SELECT *, current_timestamp() AS _ingestion_bronze, sha2(concat_ws("|", ',
        array_join(
          transform(
            array_sort(map_keys(row.cfg_target.target_schema_definition)),
            col_name -> concat('CAST(`', col_name, '` AS STRING)')
          ),
          ', '
        ),
        '), 256) AS _hashkey FROM ( <base_query> )'
      );

      -- Build and execute DDL for schema creation
      SET ddl_schema = concat('CREATE SCHEMA IF NOT EXISTS lakehouse_{env}.', row.cfg_target.target_schema);
      EXECUTE IMMEDIATE ddl_schema;

      -- Build fully qualified target table name
      SET target_table_fqn = concat('lakehouse_{env}.', row.cfg_target.target_schema, '.', row.cfg_target.target_object);

      -- SELECT base_query, hash_query, query, ddl_table, dml_table, dml_max, dml_count_query, dml_count, dml_range, step;
      -- LEAVE iterConf;

      SET dml_max = 500000;
      SET dml_count_query = concat('SELECT COUNT(1) FROM (', base_query, ')');
      EXECUTE IMMEDIATE dml_count_query INTO dml_count;
      IF dml_count % dml_max = 0 THEN
        SET dml_range = dml_count/dml_max;
      ELSE
        SET dml_range = (dml_count/dml_max) + 1;
      END IF;

      iterDML: WHILE step < dml_range DO 

        SET query = replace(
          hash_query, 
          '<base_query>', 
          concat(base_query, ' ORDER BY ', row.cfg_source.watermark_column, ' LIMIT ', dml_max, ' OFFSET ', step*dml_max)
        );

        -- Check if table exists in information_schema
        IF (
            SELECT COUNT(1) 
            FROM IDENTIFIER('lakehouse_{env}.information_schema.tables')
            WHERE 
                  table_catalog = 'lakehouse_{env}'
              AND table_schema = concat("'", row.cfg_target.target_object, "'")
              AND table_name = concat("'", row.cfg_target.target_object, "'")
        )=0 AND step = 1 THEN
          -- Create table if not exists as query
          SET ddl_table = concat('CREATE TABLE IF NOT EXISTS ', target_table_fqn, ' AS SELECT * FROM (', query, ') WHERE 1=2');
          EXECUTE IMMEDIATE ddl_table;
        END IF;

        SET dml_table = concat('INSERT INTO ', target_table_fqn, ' ', query);
        EXECUTE IMMEDIATE dml_table;
        SET step = step + 1;
      END WHILE iterDML;

      IF upper(row.cfg_source.load_mode) = 'WATERMARK' THEN
        -- Update watermark_initial_value in cfg_source
        EXECUTE IMMEDIATE dml_watermark;
      END IF;

      -- SELECT ddl_table, dml_watermark, dml_table, base_query, hash_query, query, dml_count_query, dml_count, dml_max, dml_range, step;
      -- LEAVE iterConf;

    END CASE;

    INSERT INTO platform_{env}.control.tmp_result VALUES(row.cfg_config.config_id, columns, query, ddl_table, dml_table, dml_watermark, dml_count);

  END FOR iterConf;

  -- Output results
  SELECT * FROM platform_{env}.control.tmp_result ORDER BY config_id;

END;
''')

In [0]:
# Optional smoke test:
display(spark.sql(f'''
  CALL platform_{env}.utils.proc_run_bronze_scd1(
    'hswim_veiculo',
    'BRONZE',
    'cyclops'
  )
''')
)
